## 🇺🇦 Wartime Civilian Harms and Adolescent Trauma in Ukraine

_WIP - NOT FOR DISTRIBUTION_

**Preregistration and STROBE checklist in progress**

*Provided in the spirit of open science. This notebook offers a "one-click" replication wrangling and building the _oblast_ (region) and _raion_ (district) Bellingcat OSINT [Civilian Harm in Ukraine](https://ukraine.bellingcat.com/) geocoded event-level data $\rightarrow$ Ukraine Longitudinal Study (ULS) and (non-replicable) individual-level _Ukraine Longitudinal Survey_ (ULS) 1:$n$ merge. To protect the safety and anonymity of the child-adolescent ULS respondents, those data are private and cannot be released. 

⛏️ `uls_scratchpad.ipynb`<br>
Simone J. Skeen x Claude Code (06-18-2026)

1. [Prepare](#1-prepare)
2. [Import + transform](#2-import--transform-bellingcat-osint-civilian-harm-in-ukraine)


### 1. Prepare
Imports requisite packages; customizes outputs.
***
**Dependencies:** Install via `pip install -r requirements.txt` from project root before running.

In [1]:
%%capture

%pip install -r ../../requirements.txt

In [2]:
# Standard library
import json
import os
import urllib.request
import warnings
from datetime import datetime
from pathlib import Path
from time import sleep

# Third-party
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dotenv import load_dotenv
from geopy.geocoders import Nominatim
from IPython.core.interactiveshell import InteractiveShell

In [3]:
# Env variables
load_dotenv()
BELLINGCAT_API_URL = os.getenv('BELLINGCAT_API_URL')

# Output preferences
InteractiveShell.ast_node_interactivity = 'all'

pd.options.mode.copy_on_write = True
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

for category in (FutureWarning, UserWarning):
    warnings.simplefilter(action='ignore', category=category)

In [ ]:
%%script false --no-raise-error

# Project directory structure
.
└── ukraine_longitudinal_survey/
    ├── config
    ├── data/
    │   ├── raw/
    │   │   ├── level_1
    │   │   └── level_2
    │   └── processed
    ├── src/
    │   └── notebooks
    └── outputs/
        └── figures

In [4]:
# Set working directory to project root; define data paths
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('../..')
elif os.path.basename(os.getcwd()) == 'src':
    os.chdir('..')

# Inputs subdirectories
DATA_RAW = 'data/raw'
DATA_LVL1 = f'{DATA_RAW}/level_1'   ### ULS child-adolescent survey data; not for public use
DATA_LVL2 = f'{DATA_RAW}/level_2'   ### Bellingcat geotagged OSINT data 

In [5]:
# Bellingcat data source
BELLINGCAT_CSV = 'ukr-civharm-2026-01-09.csv'

# ULS merge params
SURVEY_START_DATE = '2025-04-08' ### earliest observation in ULS survey
DATE_FORMAT_INPUT = '%m/%d/%Y'   ### format in source data (if .CSV fallback)
DATE_FORMAT_ISO = '%Y-%m-%d'     ### ISO 8601 for internal use

# geocoding
NOMINATIM_USER_AGENT = 'ukraine_postcode_geocoder'
NOMINATIM_DELAY_SEC = 1  ### 1-second delay; ensures rate limit compliance

In [ ]:
%%script false --no-raise-error
#############################################################################################

# Fetch routinely updated .JSON from Bellingcat API endpoint

### docs: https://github.com/bellingcat/ukraine-timemap

def fetch_bellingcat_json(url):
    """
    Fetches Bellingcat civilian harm data from API endpoint.
    Returns list of event dictionaries.
    """
    with urllib.request.urlopen(url) as response:
        data = json.loads(response.read().decode('utf-8'))
    return data

def convert_to_csv_format(events):
    """
    Converts Bellingcat API JSON to CSV format matching ukr-civharm-*.csv structure.
    
    JSON format: id, date (YYYY-MM-DD), latitude, longitude, location, 
                 description, sources (array), impact (array), weapon_system (array)
    CSV format:  id, date (MM/DD/YYYY), latitude, longitude, location,
                 description, sources (comma-sep), associations (formatted string)
    """
    rows = []
    for event in events:
        # Convert date: YYYY-MM-DD → MM/DD/YYYY
        date_iso = event.get('date', '')
        try:
            date_obj = datetime.strptime(date_iso, '%Y-%m-%d')
            date_formatted = date_obj.strftime('%m/%d/%Y')
        except ValueError:
            date_formatted = date_iso
        
        # Join sources array
        sources = event.get('sources', [])
        sources_str = ','.join(sources) if sources else ''
        
        # Build `associations` string from `impact` & `weapon_system`
        associations_parts = []
        for impact in event.get('impact', []):
            associations_parts.append(f'Type of area affected={impact}')
        for weapon in event.get('weapon_system', []):
            associations_parts.append(f'Weapon System={weapon}')
        associations_str = ','.join(associations_parts) if associations_parts else ''
        
        rows.append({
            'id': event.get('id', ''),
            'date': date_formatted,
            'latitude': event.get('latitude', ''),
            'longitude': event.get('longitude', ''),
            'location': event.get('location', '').strip(),
            'description': event.get('description', '').strip(),
            'sources': sources_str,
            'associations': associations_str,
        })
    
    return pd.DataFrame(rows)

# Fetch
print(f"Fetching data from Bellingcat API...")
d_lvl2_raw_json = fetch_bellingcat_json(BELLINGCAT_API_URL)

# Save raw .JSON 
json_path = f'{DATA_LVL2}/ukr-civharm-{datetime.now().strftime('%Y-%m-%d')}.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(d_lvl2_raw_json, f, ensure_ascii=False, indent=2)
print(f"Raw JSON saved to: {json_path}")

# Convert & save to .CSV
d_lvl2_raw_csv = convert_to_csv_format(d_lvl2_raw_json)
csv_path = f'{DATA_LVL2}/ukr-civharm-{datetime.now().strftime('%Y-%m-%d')}.csv'
d_lvl2_raw_csv.to_csv(csv_path, index=False)

print(f"Fetched {len(d_lvl2_raw_csv):,} events")
print(f"Saved to: {csv_path}")
d_lvl2_raw_csv.head(3)
#############################################################################################

### 2. Import + transform: Bellingcat OSINT Civilian Harm in Ukraine
Imports, cleans, describes level-2 aggregate conflict data. Acquired via .csv export: https://ukraine.bellingcat.com/.

In [ ]:
#############################################################################################
d = pd.read_csv(f'data/ukr-civharm-2026-01-09.csv')
d.head(5)
#############################################################################################

,id,date,latitude,longitude,location,description,sources,associations
0,CIV0098,02/24/2022,49.212119,38.905921,NaN,"Individual injured by shelling, ambulance resp...",https://www.facebook.com/story.php?story_fbid=...,"Type of area affected=Residential,Weapon Syste..."
1,CIV0013,02/24/2022,48.055395,37.778300,NaN,Apparent strike on hospital in separatist held...,https://twitter.com/City_Donetsk/status/149687...,"Type of area affected=Healthcare,Weapon System..."
2,CIV0004,02/24/2022,47.775609,37.239673,NaN,"Explosion in central Kyiv, nothing further yet.",https://twitter.com/N_Waters89/status/14968566...,"Type of area affected=Healthcare,Weapon System..."
3,CIV0011,02/24/2022,50.308310,34.880702,NaN,Civilian buildings damaged and destroyed by sh...,https://t.me/nexta_live/16696,"Type of area affected=Commercial,Weapon System..."
4,CIV0008,02/24/2022,47.117099,37.684831,NaN,civilian homes hit by artillery.,"https://t.me/mariupolrada/8465,https://twitter...","Type of area affected=Residential,Weapon Syste..."


In [8]:
# Dupe raw for processing
#d = d_lvl2_raw_csv.copy()

# Add ascending numerical index
d['index'] = range(len(d))
d = d.set_index('index')

# Drop imprecise OS location col
d = d.drop(
    'location', 
    axis = 1, 
    errors = 'ignore',
    )

# Restrict to obs on or before ULS start date
d['date'] = pd.to_datetime(
    d['date'], 
    format = DATE_FORMAT_INPUT,
    errors = 'coerce',    
    )

uls_startdate = pd.to_datetime(SURVEY_START_DATE)
d = d[d['date'] <= uls_startdate]

# Inspect & verify
d.shape
d.info()
d.head(2)
d.tail(2)

(2446, 7)

<class 'pandas.core.frame.DataFrame'>
Index: 2446 entries, 0 to 2445
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id            2446 non-null   object        
 1   date          2446 non-null   datetime64[ns]
 2   latitude      2444 non-null   float64       
 3   longitude     2444 non-null   float64       
 4   description   2446 non-null   object        
 5   sources       2445 non-null   object        
 6   associations  2405 non-null   object        
dtypes: datetime64[ns](1), float64(2), object(4)
memory usage: 152.9+ KB


,id,date,latitude,longitude,description,sources,associations
index,,,,,,,
0,CIV0098,2022-02-24,49.212119,38.905921,"Individual injured by shelling, ambulance resp...",https://www.facebook.com/story.php?story_fbid=...,"Type of area affected=Residential,Weapon Syste..."
1,CIV0013,2022-02-24,48.055395,37.778300,Apparent strike on hospital in separatist held...,https://twitter.com/City_Donetsk/status/149687...,"Type of area affected=Healthcare,Weapon System..."


,id,date,latitude,longitude,description,sources,associations
index,,,,,,,
2444,S23173,2025-04-07,48.507810,37.747111,At least one person killed and two injured inc...,"https://t.me/astrapress/78387,https://t.me/ast...","Type of area affected=Residential,Type of area..."
2445,W4DR1R,2025-04-08,50.775975,35.251465,Heavily damaged residential buildings followin...,"https://t.me/suspilnesumy/32400,https://t.me/s...",Type of area affected=Residential


In [9]:
# dummy code: area type affected

### TODO 6/18: Loop over _all_ `associations` and `weapons_systems` 

# residential areas

d['rsdn'] = d['associations'].str.contains(
    r'Type of area affected=Residential',
    case = False,
    na = False,
    regex = True,
).astype(int)

# schools / childcare facilities

d['schl'] = d['associations'].str.contains(
    r'Type of area affected=School or childcare',
    case = False,
    na = False,
    regex = True,
).astype(int)

# verify 

print("Area type affected counts:")
print(f"  Residential: {d['rsdn'].sum()}")
print(f"  School/childcare: {d['schl'].sum()}")
print(f"\nSample rows with schl=1:")
d[d['schl'] == 1][['id', 'associations', 'rsdn', 'schl']].head(3)

Area type affected counts:
  Residential: 1094
  School/childcare: 326

Sample rows with schl=1:


,id,associations,rsdn,schl
index,,,,
13,CIV1921,"Type of area affected=School or childcare,Weap...",0,1
17,CIV0378,"Type of area affected=School or childcare,Weap...",0,1
29,CIV0024,"Type of area affected=School or childcare,Weap...",0,1


In [ ]:
#############################################################################################
d.head(5)
#############################################################################################

,id,date,latitude,longitude,description,sources,associations,rsdn,schl
index,,,,,,,,,
0,CIV0098,2022-02-24,49.212119,38.905921,"Individual injured by shelling, ambulance resp...",https://www.facebook.com/story.php?story_fbid=...,"Type of area affected=Residential,Weapon Syste...",1,0
1,CIV0013,2022-02-24,48.055395,37.778300,Apparent strike on hospital in separatist held...,https://twitter.com/City_Donetsk/status/149687...,"Type of area affected=Healthcare,Weapon System...",0,0
2,CIV0004,2022-02-24,47.775609,37.239673,"Explosion in central Kyiv, nothing further yet.",https://twitter.com/N_Waters89/status/14968566...,"Type of area affected=Healthcare,Weapon System...",0,0
3,CIV0011,2022-02-24,50.308310,34.880702,Civilian buildings damaged and destroyed by sh...,https://t.me/nexta_live/16696,"Type of area affected=Commercial,Weapon System...",0,0
4,CIV0008,2022-02-24,47.117099,37.684831,civilian homes hit by artillery.,"https://t.me/mariupolrada/8465,https://twitter...","Type of area affected=Residential,Weapon Syste...",1,0


#### 2a. Reverse geocode: latitude / longitude $\rightarrow$ UA postcode
Reverse geocodes event coordinates to Ukrainian postcodes via Nominatim API.

In [12]:
#############################################################################################
# Restrict to n = 100 for geocding tests
d = d.iloc[:100]
d.info()
#############################################################################################

<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id            100 non-null    object        
 1   date          100 non-null    datetime64[ns]
 2   latitude      100 non-null    float64       
 3   longitude     100 non-null    float64       
 4   description   100 non-null    object        
 5   sources       100 non-null    object        
 6   associations  96 non-null     object        
 7   rsdn          100 non-null    int64         
 8   schl          100 non-null    int64         
dtypes: datetime64[ns](1), float64(2), int64(2), object(4)
memory usage: 7.8+ KB


In [ ]:
# Convert latitude/longitude coordinates → Ukrainian postcodes via Nominatim API
geolocator = Nominatim(user_agent = NOMINATIM_USER_AGENT)

def get_postcode(lat, lon):
    """
    Reverse geocode latitude/longitude to get postcode.
    Returns None if postcode not found.
    """
    try:
        location = geolocator.reverse(f"{lat}, {lon}", language = 'en')
        if location and location.raw.get('address'):
            postcode = location.raw['address'].get('postcode')
            return postcode
        return None
    except Exception as e:
        print(f"Error geocoding ({lat}, {lon}): {e}")
        return None

# Apply geocoding to each row with rate-limited delay
postcodes = []
total_rows = len(d)

for idx, row in d.iterrows():
    lat = row['latitude']
    lon = row['longitude']
    
    postcode = get_postcode(lat, lon)
    postcodes.append(postcode)
    
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{total_rows} rows...")
    
    sleep(NOMINATIM_DELAY_SEC)

d['postcode'] = postcodes

print(f"\nGeocoding complete!")
print(f"Postcodes found: {d['postcode'].notna().sum()}/{len(d)}")
print(f"\nSample results:")
print(d[['latitude', 'longitude', 'postcode']].head(10))

Processed 100/100 rows...

Geocoding complete!
Postcodes found: 99/100

Sample results:
        latitude  longitude postcode
index                               
0      49.212119  38.905921    92760
1      48.055395  37.778300    83054
2      47.775609  37.239673    85670
3      50.308310  34.880702    42700
4      47.117099  37.684831    87503
5      46.227914  34.642854    75565
6      48.748938  30.218889    20300
7      49.850781  36.659762    63501
8      48.051489  37.778075    83054
9      47.379841  37.756229    87100


In [14]:
# Enumerate unique postcodes in d
def count_unique_postcodes(df, col = 'postcode'):
    """
    Returns count of unique non-null values in specified column.
    """
    unique_vals = df[col].dropna().unique()
    return len(unique_vals)

n_unique = count_unique_postcodes(d)
print(f"Unique postcodes in d: {n_unique}")

Unique postcodes in d: 76


In [15]:
# Extract first 2 digits of postcode as `admin_unit` id
d['admin_unit'] = d['postcode'].astype(str).str[:2]

# Replace 'na' (from NaN conversion) with actual NaN
d.loc[d['admin_unit'] == 'na', 'admin_unit'] = np.nan

# Enumerate unique admin units
n_unique_admin = count_unique_postcodes(
    d, 
    col = 'admin_unit',
    )
print(f"Unique admin units in d: {n_unique_admin}")
print(f"\nAdmin unit distribution:")
d['admin_unit'].value_counts().sort_index()

Unique admin units in d: 26

Admin unit distribution:


admin_unit
01     1
02     1
03     1
04     5
07     2
08    11
14     4
15     1
20     1
34     1
42     4
54     1
61    28
62     2
63     2
72     1
73     2
74     2
75     1
83     6
84     3
85     5
87     6
92     4
93     4
No     1
Name: count, dtype: int64

In [16]:
# Map `admin_unit` to oblast
### Source: Ukrposhta postal code system: https://en.wikipedia.org/wiki/Postal_codes_in_Ukraine

### TODO 2/2: Manual inspect for `admin_unit`:`oblast` mapping verification
### TODO 6/18: Get these messy strings out of here...

ADMIN_UNIT_TO_OBLAST = {
    # Kyiv city (01-06)
    '01': 'Kyiv', '02': 'Kyiv', '03': 'Kyiv', 
    '04': 'Kyiv', '05': 'Kyiv', '06': 'Kyiv',
    # Kyiv Oblast (07-09)
    '07': 'Kyiv Oblast', '08': 'Kyiv Oblast', '09': 'Kyiv Oblast',
    # Zhytomyr Oblast (10-13)
    '10': 'Zhytomyr Oblast', '11': 'Zhytomyr Oblast', 
    '12': 'Zhytomyr Oblast', '13': 'Zhytomyr Oblast',
    # Chernihiv Oblast (14-17)
    '14': 'Chernihiv Oblast', '15': 'Chernihiv Oblast',
    '16': 'Chernihiv Oblast', '17': 'Chernihiv Oblast',
    # Cherkasy Oblast (18-22)
    '18': 'Cherkasy Oblast', '19': 'Cherkasy Oblast', '20': 'Cherkasy Oblast',
    '21': 'Cherkasy Oblast', '22': 'Cherkasy Oblast',
    # Vinnytsia Oblast (23-24)
    '23': 'Vinnytsia Oblast', '24': 'Vinnytsia Oblast',
    # Kirovohrad Oblast (25-28)
    '25': 'Kirovohrad Oblast', '26': 'Kirovohrad Oblast',
    '27': 'Kirovohrad Oblast', '28': 'Kirovohrad Oblast',
    # Khmelnytskyi Oblast (29-32)
    '29': 'Khmelnytskyi Oblast', '30': 'Khmelnytskyi Oblast',
    '31': 'Khmelnytskyi Oblast', '32': 'Khmelnytskyi Oblast',
    # Rivne Oblast (33-35)
    '33': 'Rivne Oblast', '34': 'Rivne Oblast', '35': 'Rivne Oblast',
    # Poltava Oblast (36-39)
    '36': 'Poltava Oblast', '37': 'Poltava Oblast',
    '38': 'Poltava Oblast', '39': 'Poltava Oblast',
    # Sumy Oblast (40-42)
    '40': 'Sumy Oblast', '41': 'Sumy Oblast', '42': 'Sumy Oblast',
    # Volyn Oblast (43-45)
    '43': 'Volyn Oblast', '44': 'Volyn Oblast', '45': 'Volyn Oblast',
    # Ternopil Oblast (46-48)
    '46': 'Ternopil Oblast', '47': 'Ternopil Oblast', '48': 'Ternopil Oblast',
    # Dnipropetrovsk Oblast (49-53)
    '49': 'Dnipropetrovsk Oblast', '50': 'Dnipropetrovsk Oblast',
    '51': 'Dnipropetrovsk Oblast', '52': 'Dnipropetrovsk Oblast',
    '53': 'Dnipropetrovsk Oblast',
    # Mykolaiv Oblast (54-57)
    '54': 'Mykolaiv Oblast', '55': 'Mykolaiv Oblast',
    '56': 'Mykolaiv Oblast', '57': 'Mykolaiv Oblast',
    # Chernivtsi Oblast (58-60)
    '58': 'Chernivtsi Oblast', '59': 'Chernivtsi Oblast', '60': 'Chernivtsi Oblast',
    # Kharkiv Oblast (61-64)
    '61': 'Kharkiv Oblast', '62': 'Kharkiv Oblast',
    '63': 'Kharkiv Oblast', '64': 'Kharkiv Oblast',
    # Odesa Oblast (65-68)
    '65': 'Odesa Oblast', '66': 'Odesa Oblast',
    '67': 'Odesa Oblast', '68': 'Odesa Oblast',
    # Zaporizhzhia Oblast (69-72)
    '69': 'Zaporizhzhia Oblast', '70': 'Zaporizhzhia Oblast',
    '71': 'Zaporizhzhia Oblast', '72': 'Zaporizhzhia Oblast',
    # Kherson Oblast (73-75)
    '73': 'Kherson Oblast', '74': 'Kherson Oblast', '75': 'Kherson Oblast',
    # Ivano-Frankivsk Oblast (76-78)
    '76': 'Ivano-Frankivsk Oblast', '77': 'Ivano-Frankivsk Oblast',
    '78': 'Ivano-Frankivsk Oblast',
    # Lviv Oblast (79-82)
    '79': 'Lviv Oblast', '80': 'Lviv Oblast',
    '81': 'Lviv Oblast', '82': 'Lviv Oblast',
    # Donetsk Oblast (83-87)
    '83': 'Donetsk Oblast', '84': 'Donetsk Oblast', '85': 'Donetsk Oblast',
    '86': 'Donetsk Oblast', '87': 'Donetsk Oblast',
    # Zakarpattia Oblast (88-90)
    '88': 'Zakarpattia Oblast', '89': 'Zakarpattia Oblast', '90': 'Zakarpattia Oblast',
    # Luhansk Oblast (91-94)
    '91': 'Luhansk Oblast', '92': 'Luhansk Oblast',
    '93': 'Luhansk Oblast', '94': 'Luhansk Oblast',
    # AR Crimea (95-98) & Sevastopol (99)
    '95': 'AR Crimea', '96': 'AR Crimea', '97': 'AR Crimea', '98': 'AR Crimea',
    '99': 'Sevastopol',
    }

# Map `admin_unit` → `oblast`
d['oblast'] = d['admin_unit'].map(ADMIN_UNIT_TO_OBLAST)

# Verify mapping
print(f"Mapped oblasts: {d['oblast'].notna().sum()}/{len(d)}")
print(f"Unmapped admin units: {d[d['oblast'].isna()]['admin_unit'].unique()}")
print(f"\nOblast distribution:")
d['oblast'].value_counts()

Mapped oblasts: 99/100
Unmapped admin units: ['No']

Oblast distribution:


oblast
Kharkiv Oblast         32
Donetsk Oblast         20
Kyiv Oblast            13
Luhansk Oblast          8
Kyiv                    8
Kherson Oblast          5
Chernihiv Oblast        5
Sumy Oblast             4
Cherkasy Oblast         1
Zaporizhzhia Oblast     1
Rivne Oblast            1
Mykolaiv Oblast         1
Name: count, dtype: int64

### PRELIM: Encode _raions_ two ways

In [ ]:
# Map latitude/longitude coordinates → Ukrainian raions via Nominatim API

In [ ]:
# Map postcodes → Ukrainian raions via Ministry of Community and Territorial Development of Ukraine open data

### 3. Collapse / export: Bellingcat OSINT Civilian Harm in Ukraine
Aggregates event-level counts at level-2 oblast level. Exports `d_uls_lvl2.csv` for 1:_n_ merge to `d_uls_lvl1.dta` (derived in Stata).

In [ ]:
# extract `admin_unit` mapping _prior_ to aggregation

admin_units_map = d_uls_lvl2.groupby('oblast')['admin_unit'].apply(
    lambda i: ', '.join(sorted(i.dropna().unique()))
).reset_index().rename(columns = {'admin_unit': 'admin_units'})

# aggregate counts

d_uls_lvl2 = d_uls_lvl2.groupby('oblast', as_index = False).agg({
    'id': 'count',
    'rsdn': 'sum',
    'schl': 'sum',
    })

# rename cols

d_uls_lvl2 = d_uls_lvl2.rename(columns = {'id': 'n_events'},)

# merge admin units

d_uls_lvl2 = d_uls_lvl2.merge(
    admin_units_map, 
    on = 'oblast', 
    how = 'left',
    )

# reorder cols / sort by total events (descending)

d_uls_lvl2 = d_uls_lvl2[[
    'oblast', 
    'admin_units', 
    'n_events', 
    'rsdn', 
    'schl',
    ]]

d_uls_lvl2 = d_uls_lvl2.sort_values(
    'n_events', 
    ascending = False,
    ).reset_index(drop = True)

# save

d_uls_lvl2.to_csv(PROJECT_ROOT / 'data' / 'd_uls_lvl2.csv', index = False)

print(f"Oblast-level aggregation: {len(d_uls_lvl2)} oblasts")
print(f"Saved to: {PROJECT_ROOT / 'data' / 'd_uls_lvl2.csv'}")
d_uls_lvl2